In [1]:
import json
import tiktoken
import os
from pathlib import Path
import requests
from dotenv import load_dotenv
import re
import time

In [2]:
def load_query_data(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

In [3]:
def parse_llm_response(response_content: str, options_format) -> dict:
    """
    Function to parse the raw LLM response into a structured format.
    """
    content = response_content.strip()
    if options_format == "letters":
        valid_options = ['A', 'B']
        pattern = r'[ABC]'
    elif options_format == "numbers":
        valid_options = ['1', '2', '3']
        pattern = r'[123]'
    else:
        raise ValueError("Unsupported options format")

    # Single character response
    if len(content) == 1:
        if content in valid_options:
            return {'parsed_choice': content, 'parse_method': 'single_char'}
        else: 
            return {'parsed_choice': None, 'parse_method': 'single_char_invalid'}
    else:
        # Checking for final answer tag - take LAST match to skip stray letters in reasoning
        final_answer_matches = re.findall(r'(?:\*\*)?Final Answer(?:\*\*)?[:\s]*({})'.format(pattern), content, re.IGNORECASE)
        if final_answer_matches:
            return {'parsed_choice': final_answer_matches[-1].upper(), 'parse_method': 'final_answer_tag'}
        
        answer_is_match = re.search(r'(?:best\s+)?answer\s+is[:\s]*({})'.format(pattern), content, re.IGNORECASE)
        if answer_is_match:
            return {'parsed_choice': answer_is_match.group(1).upper(), 'parse_method': 'answer_is'}

        # Checking for bolded response
        bold_match = re.search(rf'\*\*({pattern})\*\*(?!\.)', content)       
        if bold_match:
            return {'parsed_choice': bold_match.group(1).upper(), 'parse_method': 'bold'}
        
        # Multi character longer response
        # Checking for single characters first
        matches = re.findall(rf'\b({pattern})\b', content)
        if len(matches) > 1:
            # Return the first valid match
            return {'parsed_choice': matches[0].upper(), 'parse_method': 'multi_char_first'}
        elif len(matches) == 1:
            return {'parsed_choice': matches[0].upper(), 'parse_method': 'multi_char_single_match'}
        elif len(matches) == 0:
            # Checking for single character cap at the start or the end of the response content
            if content[0] in valid_options:
                return {'parsed_choice': content[0], 'parse_method': 'start_char'}
            elif content[-1] in valid_options:
                return {'parsed_choice': content[-1], 'parse_method': 'end_char'}
            else:
                return {'parsed_choice': None, 'parse_method': 'no_valid_choice'}

In [4]:
def parse_all_responses(results: dict, options_format: str) -> dict:
    """
    Function to parse all LLM responses in the results dict.
    """
    for item in results['results']: 
        for response in item['responses']:
            parsed = parse_llm_response(response['content'], options_format)
            response['parsed_choice'] = parsed['parsed_choice']
            response['parse_method'] = parsed['parse_method']
    return results

In [5]:
def check_parse_methods(response_file_path: str, options_format:   
  str = "letters"):                                                  
      results = load_query_data(response_file_path)                  
      parsed = parse_all_responses(results, options_format)          
                                                                     
      methods = {}                                                   
      none_count = 0                                                 
      for item in parsed['results']:                                 
          for response in item['responses']:                         
              method = response.get('parse_method', 'unknown')       
              parsed_choice = response.get('parsed_choice')          
              if parsed_choice is None:                              
                  none_count += 1                                    
              methods[method] = methods.get(method, 0) + 1           
                                                                     
      print(f"File: {response_file_path.split('/')[-1]}")            
      print(f"  Parse methods: {methods}")                           
      print(f"  None/unparsed: {none_count}")                        
      print()                                                                                                      

In [6]:
def compute_choice_distribution(results: dict, options_format: str) -> dict:
   """
    Returns dictionary of choice distribution across all responses. 
    {
     gt_distribution: {
        'A': count,
        'B': count,
        ...
     }
     response_choice_distribution: {
        'A': count,
        'B': count,
        ...
    }
    }
    Counts each sample independently.
    """

   if options_format == "letters":
        position_to_choice = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
        gt_counts = {'A': 0, 'B': 0, 'C': 0} 
        model_counts = {'A': 0, 'B': 0, 'C': 0} 
   elif options_format == "numbers":
        position_to_choice = {0: '1', 1: '2', 2: '3', 3: '4', 4: '5'}
        gt_counts = {'1': 0, '2': 0, '3': 0} 
        model_counts = {'1': 0, '2': 0, '3': 0} 
   else:
        raise ValueError("Unsupported options format")                                      
                             
    
   for item in results['results']:
         gt_position = item['gt_position']
         gt_choice = position_to_choice[gt_position]
         for response in item['responses']:
              gt_counts[gt_choice]  += 1

              parsed_choice = response.get('parsed_choice')
              if parsed_choice in model_counts:
                   model_counts[parsed_choice] += 1
   
   gt_total = sum(gt_counts.values())
   model_total = sum(model_counts.values())
   print(f"GT Total: {gt_total}, Model Total: {model_total}")

   distribution_dict = {
         "gt_distribution": gt_counts,
         "response_choice_distribution": model_counts
   } 
   return distribution_dict

In [7]:
def compute_accuracy_by_position(parsed_results):                   
      """Compute accuracy separately for GT at A, B, C"""             
      position_stats = {0: {'correct': 0, 'total': 0},                
                        1: {'correct': 0, 'total': 0},                
                        2: {'correct': 0, 'total': 0}}                
      position_to_choice = {0: 'A', 1: 'B', 2: 'C'}                   
                                                                      
      for item in parsed_results['results']:
        gt_pos = item['gt_position']                                
        gt_choice = position_to_choice[gt_pos]                      
        choices = []                                                            
        for response in item['responses']:                          
              parsed = response.get('parsed_choice')
              choices.append(parsed) 
        if choices:
              majority = max(set(choices), key=choices.count)
              position_stats[gt_pos]['total'] += 1
              if majority == gt_choice:
                   position_stats[gt_pos]['correct'] += 1       
                                                                      
      return position_stats

In [8]:
def calculate_pairwise_gt_alignment(parsed_results, options_format="letters"):
    """                                               
    For each GT-distractor pair, checks if the model picks GT in both orderings, neither, or flips. 
    Returns overall win rate, loss rate, flip rate, and ID lists.                        
    """ 
    if options_format == "letters":
        position_to_choice = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
    elif options_format == "numbers":
        position_to_choice = {0: '1', 1: '2', 2: '3', 3: '4', 4: '5'}
    else:
        raise ValueError("Unsupported options format")
    
    # Group queries by base pair ID
    pair_groups = {}
    for item in parsed_results['results']:
        query_id = item['query_id']
        if '_comp_pos' in query_id:
            base_id = query_id.rsplit('_comp_pos', 1)[0]
            if base_id not in pair_groups:
                pair_groups[base_id] = []
            pair_groups[base_id].append(item)
    print(f"Total pairs after base_id grouping: {len(pair_groups)}")
    
    win_ids = []
    loss_ids = []  
    flip_ids = []
    unparsed_ids = []

    for base_id, items in pair_groups.items():
        gt_picks = []
        for item in items:
            gt_choice = position_to_choice[item['gt_position']]

            # Majority vote across samples
            choices = [r.get('parsed_choice') for r in item['responses'] if r.get('parsed_choice')]
            if not choices: 
                gt_picks.append(None)
                continue
            majority = max(set(choices), key=choices.count)
            gt_picks.append(majority == gt_choice)

        if None in gt_picks:
            unparsed_ids.append(base_id)
        elif all(gt_picks):
            win_ids.append(base_id)
        elif not any(gt_picks):
            loss_ids.append(base_id)
        else:
            flip_ids.append(base_id)
        
    total = len(pair_groups)
    print(f"Wins: {len(win_ids)}/{total} ({len(win_ids)/total:.1%})")                          
    print(f"Losses: {len(loss_ids)}/{total}({len(loss_ids)/total:.1%})")
    print(f"Flips: {len(flip_ids)}/{total}({len(flip_ids)/total:.1%})")          
    print(f"Unparsed: {len(unparsed_ids)}/{total}")
    
    return {
        "total_pairs": total,
        "wins": len(win_ids),
        "losses": len(loss_ids),
        "flips": len(flip_ids),
        "unparsed": len(unparsed_ids),
        "win_rate": len(win_ids) / total if total > 0 else 0.0,
        "loss_rate": len(loss_ids) / total if total > 0 else 0.0,
        "flip_rate": len(flip_ids) / total if total > 0 else 0.0,
        "win_ids": win_ids,
        "loss_ids": loss_ids,
        "flip_ids": flip_ids,
        "unparsed_ids": unparsed_ids 
    }

In [9]:
gemini_flash_file = '../primary_queries_and_responses/main_responses/gemini-3-flash.json'
gemini_flash_results = load_query_data(gemini_flash_file)
parsed_gemini_flash = parse_all_responses(gemini_flash_results, options_format="letters")

# Check Parse Methods
check_parse_methods(gemini_flash_file, options_format="letters")

# Choice Distribution
distribution = compute_choice_distribution(parsed_gemini_flash, options_format="letters")
print(f"GT Distribution: {distribution['gt_distribution']}")
print(f"Model Response Distribution: {distribution['response_choice_distribution']}")

# Accuracy by GT Position
gemini_flash_by_pos = compute_accuracy_by_position(parsed_gemini_flash)
for pos in [0, 1]:
    letter = ['A', 'B'][pos]
    stats = gemini_flash_by_pos[pos]
    acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"GT at {letter}: {stats['correct']}/{stats['total']} = {acc:.1%}")

# Alignment by Position
gemini_flash_alignment = calculate_pairwise_gt_alignment(parsed_gemini_flash, options_format="letters")



File: gemini-3-flash.json
  Parse methods: {'final_answer_tag': 160}
  None/unparsed: 0

GT Total: 160, Model Total: 160
GT Distribution: {'A': 80, 'B': 80, 'C': 0}
Model Response Distribution: {'A': 79, 'B': 81, 'C': 0}
GT at A: 55/80 = 68.8%
GT at B: 56/80 = 70.0%
Total pairs after base_id grouping: 80
Wins: 50/80 (62.5%)
Losses: 19/80(23.8%)
Flips: 11/80(13.8%)
Unparsed: 0/80


In [10]:
opus_file = '../primary_queries_and_responses/main_responses/claude-opus-4.5.json'
opus_results = load_query_data(opus_file)
parsed_opus = parse_all_responses(opus_results, options_format="letters")

# Check Parse Methods
check_parse_methods(opus_file, options_format="letters")

# Choice Distribution
distribution = compute_choice_distribution(parsed_opus, options_format="letters")
print(f"GT Distribution: {distribution['gt_distribution']}")
print(f"Model Response Distribution: {distribution['response_choice_distribution']}")

# Accuracy by GT Position
parsed_opus_by_pos = compute_accuracy_by_position(parsed_opus)
for pos in [0, 1]:
    letter = ['A', 'B'][pos]
    stats = parsed_opus_by_pos[pos]
    acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"GT at {letter}: {stats['correct']}/{stats['total']} = {acc:.1%}")

# Alignment by Position
opus_alignment = calculate_pairwise_gt_alignment(parsed_opus, options_format="letters")



File: claude-opus-4.5.json
  Parse methods: {'final_answer_tag': 160}
  None/unparsed: 0

GT Total: 160, Model Total: 160
GT Distribution: {'A': 80, 'B': 80, 'C': 0}
Model Response Distribution: {'A': 69, 'B': 91, 'C': 0}
GT at A: 46/80 = 57.5%
GT at B: 57/80 = 71.2%
Total pairs after base_id grouping: 80
Wins: 44/80 (55.0%)
Losses: 21/80(26.2%)
Flips: 15/80(18.8%)
Unparsed: 0/80


In [11]:
gpt_oss_file = '../primary_queries_and_responses/main_responses/gpt-oss-120b.json'
gpt_oss_results = load_query_data(gpt_oss_file)
parsed_gpt_oss = parse_all_responses(gpt_oss_results, options_format="letters")

# Check Parse Methods
check_parse_methods(gpt_oss_file, options_format="letters")

# Choice Distribution
distribution = compute_choice_distribution(parsed_gpt_oss, options_format="letters")
print(f"GT Distribution: {distribution['gt_distribution']}")
print(f"Model Response Distribution: {distribution['response_choice_distribution']}")

# Accuracy by GT Position
parsed_gpt_oss_by_pos = compute_accuracy_by_position(parsed_gpt_oss)
for pos in [0, 1]:
    letter = ['A', 'B'][pos]
    stats = parsed_gpt_oss_by_pos[pos]
    acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"GT at {letter}: {stats['correct']}/{stats['total']} = {acc:.1%}")

# Alignment by Position
gpt_oss_alignment = calculate_pairwise_gt_alignment(parsed_gpt_oss, options_format="letters")

File: gpt-oss-120b.json
  Parse methods: {'final_answer_tag': 159, 'multi_char_single_match': 1}
  None/unparsed: 0

GT Total: 160, Model Total: 160
GT Distribution: {'A': 80, 'B': 80, 'C': 0}
Model Response Distribution: {'A': 77, 'B': 83, 'C': 0}
GT at A: 44/80 = 55.0%
GT at B: 47/80 = 58.8%
Total pairs after base_id grouping: 80
Wins: 40/80 (50.0%)
Losses: 29/80(36.2%)
Flips: 11/80(13.8%)
Unparsed: 0/80


In [12]:
haiku_file = '../primary_queries_and_responses/main_responses/claude-haiku-4.5.json'
haiku_results = load_query_data(haiku_file)
parsed_haiku = parse_all_responses(haiku_results, options_format="letters")

# Check Parse Methods
check_parse_methods(haiku_file, options_format="letters")

# Choice Distribution
distribution = compute_choice_distribution(parsed_haiku, options_format="letters")
print(f"GT Distribution: {distribution['gt_distribution']}")
print(f"Model Response Distribution: {distribution['response_choice_distribution']}")

# Accuracy by GT Position
parsed_haiku_by_pos = compute_accuracy_by_position(parsed_haiku)
for pos in [0, 1]:
    letter = ['A', 'B'][pos]
    stats = parsed_haiku_by_pos[pos]
    acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"GT at {letter}: {stats['correct']}/{stats['total']} = {acc:.1%}")

# Alignment by Position
haiku_alignment = calculate_pairwise_gt_alignment(parsed_haiku, options_format="letters")

File: claude-haiku-4.5.json
  Parse methods: {'final_answer_tag': 129, 'bold': 31}
  None/unparsed: 0

GT Total: 160, Model Total: 160
GT Distribution: {'A': 80, 'B': 80, 'C': 0}
Model Response Distribution: {'A': 61, 'B': 99, 'C': 0}
GT at A: 42/80 = 52.5%
GT at B: 61/80 = 76.2%
Total pairs after base_id grouping: 80
Wins: 40/80 (50.0%)
Losses: 17/80(21.2%)
Flips: 23/80(28.7%)
Unparsed: 0/80


In [13]:
qwen_file = '../primary_queries_and_responses/main_responses/qwen3-max-thinking.json'
qwen_results = load_query_data(qwen_file)
parsed_qwen = parse_all_responses(qwen_results, options_format="letters")

# Check Parse Methods
check_parse_methods(qwen_file, options_format="letters")

# Choice Distribution
distribution = compute_choice_distribution(parsed_qwen, options_format="letters")
print(f"GT Distribution: {distribution['gt_distribution']}")
print(f"Model Response Distribution: {distribution['response_choice_distribution']}")

# Accuracy by GT Position
parsed_qwen_by_pos = compute_accuracy_by_position(parsed_qwen)
for pos in [0, 1]:
    letter = ['A', 'B'][pos]
    stats = parsed_qwen_by_pos[pos]
    acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"GT at {letter}: {stats['correct']}/{stats['total']} = {acc:.1%}")

# Alignment by Position
qwen_alignment = calculate_pairwise_gt_alignment(parsed_qwen, options_format="letters")

File: qwen3-max-thinking.json
  Parse methods: {'final_answer_tag': 160}
  None/unparsed: 0

GT Total: 160, Model Total: 160
GT Distribution: {'A': 80, 'B': 80, 'C': 0}
Model Response Distribution: {'A': 61, 'B': 99, 'C': 0}
GT at A: 41/80 = 51.2%
GT at B: 60/80 = 75.0%
Total pairs after base_id grouping: 80
Wins: 40/80 (50.0%)
Losses: 19/80(23.8%)
Flips: 21/80(26.2%)
Unparsed: 0/80


In [14]:
gemini_pro_file = '../primary_queries_and_responses/main_responses/gemini-3-pro.json'
gemini_pro_results = load_query_data(gemini_pro_file)
parsed_gemini_pro = parse_all_responses(gemini_pro_results, options_format="letters")

# Check Parse Methods
check_parse_methods(gemini_pro_file, options_format="letters")

# Choice Distribution
distribution = compute_choice_distribution(parsed_gemini_pro, options_format="letters")
print(f"GT Distribution: {distribution['gt_distribution']}")
print(f"Model Response Distribution: {distribution['response_choice_distribution']}")

# Accuracy by GT Position
parsed_gemini_pro_by_pos = compute_accuracy_by_position(parsed_gemini_pro)
for pos in [0, 1]:
    letter = ['A', 'B'][pos]
    stats = parsed_gemini_pro_by_pos[pos]
    acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"GT at {letter}: {stats['correct']}/{stats['total']} = {acc:.1%}")

# Alignment by Position
gemini_pro_alignment = calculate_pairwise_gt_alignment(parsed_gemini_pro, options_format="letters")

File: gemini-3-pro.json
  Parse methods: {'final_answer_tag': 160}
  None/unparsed: 0

GT Total: 160, Model Total: 160
GT Distribution: {'A': 80, 'B': 80, 'C': 0}
Model Response Distribution: {'A': 76, 'B': 84, 'C': 0}
GT at A: 59/80 = 73.8%
GT at B: 63/80 = 78.8%
Total pairs after base_id grouping: 80
Wins: 54/80 (67.5%)
Losses: 12/80(15.0%)
Flips: 14/80(17.5%)
Unparsed: 0/80


In [15]:
llama_maverick_file = '../primary_queries_and_responses/main_responses/llama-4-maverick.json'
llama_maverick_results = load_query_data(llama_maverick_file)
parsed_llama_maverick = parse_all_responses(llama_maverick_results, options_format="letters")

# Check Parse Methods
check_parse_methods(llama_maverick_file, options_format="letters")

# Choice Distribution
distribution = compute_choice_distribution(parsed_llama_maverick, options_format="letters")
print(f"GT Distribution: {distribution['gt_distribution']}")
print(f"Model Response Distribution: {distribution['response_choice_distribution']}")

# Accuracy by GT Position
parsed_llama_maverick_by_pos = compute_accuracy_by_position(parsed_llama_maverick)
for pos in [0, 1]:
    letter = ['A', 'B'][pos]
    stats = parsed_llama_maverick_by_pos[pos]
    acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"GT at {letter}: {stats['correct']}/{stats['total']} = {acc:.1%}")

# Alignment by Position
llama_maverick_alignment = calculate_pairwise_gt_alignment(parsed_llama_maverick, options_format="letters")

File: llama-4-maverick.json
  Parse methods: {'final_answer_tag': 159, 'answer_is': 1}
  None/unparsed: 0

GT Total: 160, Model Total: 160
GT Distribution: {'A': 80, 'B': 80, 'C': 0}
Model Response Distribution: {'A': 80, 'B': 80, 'C': 0}
GT at A: 44/80 = 55.0%
GT at B: 44/80 = 55.0%
Total pairs after base_id grouping: 80
Wins: 36/80 (45.0%)
Losses: 28/80(35.0%)
Flips: 16/80(20.0%)
Unparsed: 0/80


In [16]:
gpt_pro_file = '../primary_queries_and_responses/main_responses/gpt-5.2-pro.json'
gpt_pro_results = load_query_data(gpt_pro_file)
parsed_gpt_pro = parse_all_responses(gpt_pro_results, options_format="letters")

# Check Parse Methods
check_parse_methods(gpt_pro_file, options_format="letters")

# Choice Distribution
distribution = compute_choice_distribution(parsed_gpt_pro, options_format="letters")
print(f"GT Distribution: {distribution['gt_distribution']}")
print(f"Model Response Distribution: {distribution['response_choice_distribution']}")

# Accuracy by GT Position
parsed_gpt_pro_by_pos = compute_accuracy_by_position(parsed_gpt_pro)
for pos in [0, 1]:
    letter = ['A', 'B'][pos]
    stats = parsed_gpt_pro_by_pos[pos]
    acc = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"GT at {letter}: {stats['correct']}/{stats['total']} = {acc:.1%}")

# Alignment by Position
gpt_pro_alignment = calculate_pairwise_gt_alignment(parsed_gpt_pro, options_format="letters")

File: gpt-5.2-pro.json
  Parse methods: {'final_answer_tag': 160}
  None/unparsed: 0

GT Total: 160, Model Total: 160
GT Distribution: {'A': 80, 'B': 80, 'C': 0}
Model Response Distribution: {'A': 70, 'B': 90, 'C': 0}
GT at A: 46/80 = 57.5%
GT at B: 56/80 = 70.0%
Total pairs after base_id grouping: 80
Wins: 46/80 (57.5%)
Losses: 24/80(30.0%)
Flips: 10/80(12.5%)
Unparsed: 0/80


In [17]:
non_standard_methods = ['multi_char_single_match', 'multi_char_first', 'bold', 'start_char', 'end_char', 'answer_is'] 
                                                                             
non_standard_queries = []                                                          
for item in parsed_llama_maverick['results']:                                     
      for i, response in enumerate(item['responses']):                       
          if response.get('parse_method') in non_standard_methods:                         
              non_standard_queries.append({                                          
                  'query_id': item['query_id'],                              
                  'sample': i,
                  'parse_method': response.get('parse_method'),                                               
                  'parsed_choice': response.get('parsed_choice'),            
                  'content_snippet': response['content']                                                                     
              })                                                             
                                                                             
print(f"Non standard parsed responses ({len(non_standard_queries)}):")                     
for q in non_standard_queries:    
      print(f"q[{q['query_id']}] parse method: {q['parse_method']}")                                                 
      print(f"  {q['query_id']} sample {q['sample']}: chose {q['parsed_choice']}")                                                     
      print(f"    ...{q['content_snippet']}")

Non standard parsed responses (1):
q[2602.07755_fw0_d1_comp_pos1] parse method: answer_is
  2602.07755_fw0_d1_comp_pos1 sample 0: chose B
    ...## Step 1: Review the Research Goal
The research goal is to develop a framework that automatically learns memory designs for agentic systems, enabling them to continually learn from past experiences across diverse domains without relying on manual engineering of memory components.

## Step 2: Analyze the Hypotheses Already Tested and Key Results
The project has tested several hypotheses related to the effectiveness, scalability, adaptability, and cost-efficiency of learned memory designs compared to human-designed baselines. The key results have shown that learned memory designs outperform human-designed baselines in various aspects, including overall performance, scalability, adaptability under task distribution shift, and cost-efficiency. They also generalize robustly across different foundation models and specialize to the demands of differ

In [18]:
def calculate_intermodel_agreement(alignment_a,  alignment_b, model_a_name="Model A", model_b_name="Model B"):                              
      """                                               
      For each pair where both models have stable       
  preferences (no flips),                               
      checks if they agree (both win or both loss).     
      """                                               
      # Stable sets for each model (wins + losses,  excluding flips)                                      
      stable_a = set(alignment_a['win_ids'] + alignment_a['loss_ids'] + alignment_a['flip_ids'])                              
      stable_b = set(alignment_b['win_ids'] + alignment_b['loss_ids'] + alignment_b['flip_ids'])                              
                  
      # Only compare pairs where both models are stable 
      comparable = stable_a & stable_b
                                                        
      wins_a = set(alignment_a['win_ids'])              
      wins_b = set(alignment_b['win_ids'])              
      losses_a = set(alignment_a['loss_ids'])           
      losses_b = set(alignment_b['loss_ids']) 
      flips_a = set(alignment_a['flip_ids'])
      flips_b = set(alignment_b['flip_ids'])          
  
      both_win = wins_a & wins_b           
      both_loss = losses_a & losses_b
      both_flip = flips_a & flips_b
      agree = both_win | both_loss | both_flip                      
      disagree = comparable - agree
                                                        
      rate = len(agree) / len(comparable) if comparable else 0.0
                                                        
      print(f"{model_a_name} vs {model_b_name}")        
      print(f"  Comparable pairs: {len(comparable)}/{alignment_a['total_pairs']}")      
      print(f"  Agree: {len(agree)}/{len(comparable)} ({rate:.1%})")                                        
      print(f"    Both win: {len(both_win)}, Both loss: {len(both_loss)}, Both flip: {len(both_flip)}") 
      print(f"    Both win rate: {len(both_win)/len(comparable):.1%}, Both loss rate: {len(both_loss)/len(comparable):.1%}, Both flip rate: {len(both_flip)/len(comparable):.1%}")                                   
      print(f"  Disagree: {len(disagree)}/{len(comparable)}")                   
                  
      return {                                          
          "model_a": model_a_name,
          "model_b": model_b_name,
          "comparable_pairs": len(comparable),          
          "total_pairs": alignment_a['total_pairs'],
          "agree": len(agree),                          
          "disagree": len(disagree),
          "agreement_rate": rate, 
          "both_win": len(both_win),                    
          "both_loss": len(both_loss),
          "both_flip": len(both_flip),
          "both_win_rate": len(both_win) / len(comparable) if comparable else 0.0,
          "both_loss_rate": len(both_loss) / len(comparable) if comparable else 0.0,
          "both_flip_rate": len(both_flip) / len(comparable) if comparable else 0.0,
          "both_win_ids": sorted(both_win),             
          "both_loss_ids": sorted(both_loss), 
          "both_flip_ids": sorted(both_flip),         
          "disagree_ids": sorted(disagree)              
      }

In [19]:
# Creating a set of all queries where at least one model diverged stably.

# All pairs where at least one model stably diverged, with per-pair breakdown                                            
all_alignments = {                                               
      'Gemini Flash': gemini_flash_alignment,                      
      'Gemini Pro': gemini_pro_alignment,                          
      'Opus 4.5': opus_alignment,                                  
      'Haiku 4.5': haiku_alignment,
      'GPT-5.2 Pro': gpt_pro_alignment,                            
      'GPT-OSS 120B': gpt_oss_alignment,                           
      'Qwen3 Max': qwen_alignment,                                 
      'Llama Maverick': llama_maverick_alignment,                  
}                                                                
                                                                   
# Collect loss info per pair                                     
pair_loss_info = {}
for model_name, alignment in all_alignments.items():             
      for pid in alignment['loss_ids']:
          if pid not in pair_loss_info:                            
              pair_loss_info[pid] = []                             
          pair_loss_info[pid].append(model_name)                   
                                                                   
print(f"Pairs with at least one model diverging:{len(pair_loss_info)}/{80}\n")
for pid in sorted(pair_loss_info.keys()):                        
      models = pair_loss_info[pid]
      print(f"  {pid}: {len(models)}/8 models — {','.join(models)}")

from collections import Counter                                  
count_dist = Counter(len(models) for models in                   
pair_loss_info.values())                                         
print("\nDistribution of divergence counts:")                    
for k in sorted(count_dist.keys()): 
      print(f"  Exactly {k} model(s): {count_dist[k]} pairs")      
                                                                   
print("\nCumulative:")                                           
for threshold in range(1, 9):                                    
      cumulative = sum(count_dist[k] for k in count_dist if k >= threshold)                                                       
      print(f"  >= {threshold} models: {cumulative}/80") 

Pairs with at least one model diverging:45/80

  2512.22322_fw0_d0: 7/8 models — Gemini Flash,Gemini Pro,Opus 4.5,Haiku 4.5,GPT-5.2 Pro,Qwen3 Max,Llama Maverick
  2512.22322_fw0_d1: 8/8 models — Gemini Flash,Gemini Pro,Opus 4.5,Haiku 4.5,GPT-5.2 Pro,GPT-OSS 120B,Qwen3 Max,Llama Maverick
  2601.02439_fw0_d0: 1/8 models — Gemini Flash
  2601.02439_fw0_d1: 2/8 models — Haiku 4.5,GPT-OSS 120B
  2601.05175_fw0_d1: 2/8 models — Haiku 4.5,GPT-OSS 120B
  2601.05175_fw1_d0: 2/8 models — Opus 4.5,Llama Maverick
  2601.05175_fw1_d1: 3/8 models — GPT-5.2 Pro,GPT-OSS 120B,Llama Maverick
  2601.05175_fw2_d0: 4/8 models — Opus 4.5,Haiku 4.5,GPT-OSS 120B,Llama Maverick
  2601.05175_fw2_d1: 5/8 models — Gemini Pro,Opus 4.5,Haiku 4.5,GPT-OSS 120B,Llama Maverick
  2601.05175_fw3_d0: 5/8 models — Gemini Flash,Opus 4.5,Haiku 4.5,GPT-OSS 120B,Qwen3 Max
  2601.05175_fw3_d1: 5/8 models — Gemini Pro,Haiku 4.5,GPT-5.2 Pro,GPT-OSS 120B,Llama Maverick
  2601.05175_fw4_d0: 5/8 models — Gemini Flash,Haiku 4.5,GPT-5

In [20]:
models = ['Gemini Flash 3', 'Claude Opus 4.5', 'Claude Haiku 4.5', 'GPT-OSS 120B']
alignments = [gemini_flash_alignment, opus_alignment, gpt_oss_alignment, haiku_alignment]

# Gemini  Opus
gemini_opus_agreement = calculate_intermodel_agreement(gemini_flash_alignment, opus_alignment, model_a_name="Gemini Flash 3", model_b_name="Claude Opus 4.5")

# Gemini Haiku
gemini_haiku_agreement = calculate_intermodel_agreement(gemini_flash_alignment, haiku_alignment, model_a_name="Gemini Flash 3", model_b_name="Claude Haiku 4.5")

# Gemini GPT-OSS
gemini_gpt_oss_agreement = calculate_intermodel_agreement(gemini_flash_alignment, gpt_oss_alignment, model_a_name="Gemini Flash 3", model_b_name="GPT-OSS 120B")

# Opus Haiku
opus_haiku_agreement = calculate_intermodel_agreement(opus_alignment, haiku_alignment, model_a_name="Claude Opus 4.5", model_b_name="Claude Haiku 4.5")

# Opus GPT-OSS
opus_gpt_oss_agreement = calculate_intermodel_agreement(opus_alignment, gpt_oss_alignment, model_a_name="Claude Opus 4.5", model_b_name="GPT-OSS 120B")

# Haiku GPT-OSS
haiku_gpt_oss_agreement = calculate_intermodel_agreement(haiku_alignment, gpt_oss_alignment, model_a_name="Claude Haiku 4.5", model_b_name="GPT-OSS 120B")

# Qwen Flash
qwen_gemini_agreement = calculate_intermodel_agreement(qwen_alignment, gemini_flash_alignment, model_a_name="Qwen Max Thinking", model_b_name="Gemini Flash 3")

# Qwen Opus
qwen_opus_agreement = calculate_intermodel_agreement(qwen_alignment, opus_alignment, model_a_name="Qwen Max Thinking", model_b_name="Claude Opus 4.5")

# Qwen Haiku
qwen_haiku_agreement = calculate_intermodel_agreement(qwen_alignment, haiku_alignment, model_a_name="Qwen Max Thinking", model_b_name="Claude Haiku 4.5")

# Qwen GPT OSS 120B
qwen_gpt_oss_agreement = calculate_intermodel_agreement(qwen_alignment, gpt_oss_alignment, model_a_name="Qwen Max Thinking", model_b_name="GPT-OSS 120B")

# Gemini Pro Flash
gemini_pro_flash_agreement = calculate_intermodel_agreement(gemini_pro_alignment, gemini_flash_alignment, model_a_name="Gemini Pro", model_b_name="Gemini Flash 3")

# Gemini Pro Opus
gemini_pro_opus_agreement = calculate_intermodel_agreement(gemini_pro_alignment, opus_alignment, model_a_name="Gemini Pro", model_b_name="Claude Opus 4.5")

# Gemini Pro Haiku
gemini_pro_haiku_agreement = calculate_intermodel_agreement(gemini_pro_alignment, haiku_alignment, model_a_name="Gemini Pro", model_b_name="Claude Haiku 4.5")

# Gemini Pro GPT OSS 120B
gemini_pro_gpt_oss_agreement = calculate_intermodel_agreement(gemini_pro_alignment, gpt_oss_alignment, model_a_name="Gemini Pro", model_b_name="GPT-OSS 120B")

# Gemini Pro Qwen Max Thinking
gemini_pro_qwen_agreement = calculate_intermodel_agreement(gemini_pro_alignment, qwen_alignment, model_a_name="Gemini Pro", model_b_name="Qwen Max Thinking")



Gemini Flash 3 vs Claude Opus 4.5
  Comparable pairs: 80/80
  Agree: 49/80 (61.3%)
    Both win: 35, Both loss: 11, Both flip: 3
    Both win rate: 43.8%, Both loss rate: 13.8%, Both flip rate: 3.8%
  Disagree: 31/80
Gemini Flash 3 vs Claude Haiku 4.5
  Comparable pairs: 80/80
  Agree: 43/80 (53.8%)
    Both win: 34, Both loss: 7, Both flip: 2
    Both win rate: 42.5%, Both loss rate: 8.8%, Both flip rate: 2.5%
  Disagree: 37/80
Gemini Flash 3 vs GPT-OSS 120B
  Comparable pairs: 80/80
  Agree: 50/80 (62.5%)
    Both win: 35, Both loss: 13, Both flip: 2
    Both win rate: 43.8%, Both loss rate: 16.2%, Both flip rate: 2.5%
  Disagree: 30/80
Claude Opus 4.5 vs Claude Haiku 4.5
  Comparable pairs: 80/80
  Agree: 52/80 (65.0%)
    Both win: 35, Both loss: 10, Both flip: 7
    Both win rate: 43.8%, Both loss rate: 12.5%, Both flip rate: 8.8%
  Disagree: 28/80
Claude Opus 4.5 vs GPT-OSS 120B
  Comparable pairs: 80/80
  Agree: 52/80 (65.0%)
    Both win: 32, Both loss: 16, Both flip: 4
    Bot

In [21]:
# Llama Gemini Pro
llama_gemini_pro_agreement = calculate_intermodel_agreement(llama_maverick_alignment, gemini_pro_alignment, model_a_name="Llama Maverick", model_b_name="Gemini Pro")

# Llama Gemini Flash
llama_gemini_flash_agreement = calculate_intermodel_agreement(llama_maverick_alignment, gemini_flash_alignment, model_a_name="Llama Maverick", model_b_name="Gemini Flash 3")

# Llama Opus
llama_opus_agreement = calculate_intermodel_agreement(llama_maverick_alignment, opus_alignment, model_a_name="Llama Maverick", model_b_name="Claude Opus 4.5")

# Llama Haiku
llama_haiku_agreement = calculate_intermodel_agreement(llama_maverick_alignment, haiku_alignment, model_a_name="Llama Maverick", model_b_name="Claude Haiku 4.5")

# Llama GPT OSS
llama_gpt_oss_agreement = calculate_intermodel_agreement(llama_maverick_alignment, gpt_oss_alignment, model_a_name="Llama Maverick", model_b_name="GPT-OSS 120B")

# LLama Qwen Max Thinking
llama_qwen_agreement = calculate_intermodel_agreement(llama_maverick_alignment, qwen_alignment, model_a_name="Llama Maverick", model_b_name="Qwen Max Thinking")

Llama Maverick vs Gemini Pro
  Comparable pairs: 80/80
  Agree: 44/80 (55.0%)
    Both win: 31, Both loss: 10, Both flip: 3
    Both win rate: 38.8%, Both loss rate: 12.5%, Both flip rate: 3.8%
  Disagree: 36/80
Llama Maverick vs Gemini Flash 3
  Comparable pairs: 80/80
  Agree: 45/80 (56.2%)
    Both win: 31, Both loss: 12, Both flip: 2
    Both win rate: 38.8%, Both loss rate: 15.0%, Both flip rate: 2.5%
  Disagree: 35/80
Llama Maverick vs Claude Opus 4.5
  Comparable pairs: 80/80
  Agree: 54/80 (67.5%)
    Both win: 30, Both loss: 17, Both flip: 7
    Both win rate: 37.5%, Both loss rate: 21.2%, Both flip rate: 8.8%
  Disagree: 26/80
Llama Maverick vs Claude Haiku 4.5
  Comparable pairs: 80/80
  Agree: 50/80 (62.5%)
    Both win: 31, Both loss: 11, Both flip: 8
    Both win rate: 38.8%, Both loss rate: 13.8%, Both flip rate: 10.0%
  Disagree: 30/80
Llama Maverick vs GPT-OSS 120B
  Comparable pairs: 80/80
  Agree: 57/80 (71.2%)
    Both win: 30, Both loss: 20, Both flip: 7
    Both w

In [22]:
# GPT Pro Gemini Pro
gpt_pro_gemini_pro_agreement = calculate_intermodel_agreement(gpt_pro_alignment, gemini_pro_alignment, model_a_name="GPT Pro", model_b_name="Gemini Pro")

# GPT Pro Gemini Flash
gpt_pro_gemini_flash_agreement = calculate_intermodel_agreement(gpt_pro_alignment, gemini_flash_alignment, model_a_name="GPT Pro", model_b_name="Gemini Flash 3")

# GPT Pro Opus
gpt_pro_opus_agreement = calculate_intermodel_agreement(gpt_pro_alignment, opus_alignment, model_a_name="GPT Pro", model_b_name="Claude Opus 4.5")

# GPT Pro Haiku
gpt_pro_haiku_agreement = calculate_intermodel_agreement(gpt_pro_alignment, haiku_alignment, model_a_name="GPT Pro", model_b_name="Claude Haiku 4.5")

# GPT Pro GPT OSS
gpt_pro_gpt_oss_agreement = calculate_intermodel_agreement(gpt_pro_alignment, gpt_oss_alignment, model_a_name="GPT Pro", model_b_name="GPT-OSS 120B")

# GPT Pro Qwen Max Thinking
gpt_pro_qwen_agreement = calculate_intermodel_agreement(gpt_pro_alignment, qwen_alignment, model_a_name="GPT Pro", model_b_name="Qwen Max Thinking")

# GPT Pro Llama Maverick
gpt_pro_llama_agreement = calculate_intermodel_agreement(gpt_pro_alignment, llama_maverick_alignment, model_a_name="GPT Pro", model_b_name="Llama Maverick")

GPT Pro vs Gemini Pro
  Comparable pairs: 80/80
  Agree: 51/80 (63.7%)
    Both win: 39, Both loss: 10, Both flip: 2
    Both win rate: 48.8%, Both loss rate: 12.5%, Both flip rate: 2.5%
  Disagree: 29/80
GPT Pro vs Gemini Flash 3
  Comparable pairs: 80/80
  Agree: 50/80 (62.5%)
    Both win: 38, Both loss: 12, Both flip: 0
    Both win rate: 47.5%, Both loss rate: 15.0%, Both flip rate: 0.0%
  Disagree: 30/80
GPT Pro vs Claude Opus 4.5
  Comparable pairs: 80/80
  Agree: 52/80 (65.0%)
    Both win: 35, Both loss: 14, Both flip: 3
    Both win rate: 43.8%, Both loss rate: 17.5%, Both flip rate: 3.8%
  Disagree: 28/80
GPT Pro vs Claude Haiku 4.5
  Comparable pairs: 80/80
  Agree: 46/80 (57.5%)
    Both win: 33, Both loss: 10, Both flip: 3
    Both win rate: 41.2%, Both loss rate: 12.5%, Both flip rate: 3.8%
  Disagree: 34/80
GPT Pro vs GPT-OSS 120B
  Comparable pairs: 80/80
  Agree: 64/80 (80.0%)
    Both win: 38, Both loss: 21, Both flip: 5
    Both win rate: 47.5%, Both loss rate: 26.2

In [23]:
import pandas as pd

intermodel_results = [
      gemini_opus_agreement, gemini_haiku_agreement,
  gemini_gpt_oss_agreement,
      opus_haiku_agreement, opus_gpt_oss_agreement,
  haiku_gpt_oss_agreement,
      qwen_gemini_agreement, qwen_opus_agreement,
  qwen_haiku_agreement,
      qwen_gpt_oss_agreement, gemini_pro_flash_agreement,
  gemini_pro_opus_agreement,
      gemini_pro_haiku_agreement, gemini_pro_gpt_oss_agreement,
  gemini_pro_qwen_agreement,
      llama_gemini_pro_agreement, llama_gemini_flash_agreement,
  llama_opus_agreement,
      llama_haiku_agreement, llama_gpt_oss_agreement,
  llama_qwen_agreement,
      gpt_pro_gemini_pro_agreement, gpt_pro_gemini_flash_agreement,
  gpt_pro_opus_agreement,
      gpt_pro_haiku_agreement, gpt_pro_gpt_oss_agreement,
  gpt_pro_qwen_agreement,
      gpt_pro_llama_agreement,
  ]

records = []
for res in intermodel_results:
      records.append({
          "pair": f"{res['model_a']} -- {res['model_b']}",
          "agreement": res["agreement_rate"],
          "shared_divergence": res["both_loss_rate"],
          "shared_flip": res["both_flip_rate"],
      })

df = pd.DataFrame(records)

def bucketize(col):
      mean = df[col].mean()
      std = df[col].std(ddof=0)
      print(f"{col}: mean={mean:.3f}, std={std:.3f}")
      def label(val):
          if val >= mean + 2*std:
              return "> +2σ"
          if val >= mean + 1*std:
              return "+1σ to +2σ"
          if val >= mean:
              return "≥ mean"
          if val <= mean - 2*std:
              return "< -2σ"
          if val <= mean - 1*std:
              return "-1σ to -2σ"
          return "< mean"
      df[f"{col}_bucket"] = df[col].apply(label)

for col in ["agreement", "shared_divergence"]:
      bucketize(col)

display(df.sort_values("agreement", ascending=False))

agreement: mean=0.619, std=0.060
shared_divergence: mean=0.156, std=0.044


,pair,agreement,shared_divergence,shared_flip,agreement_bucket,shared_divergence_bucket
25,GPT Pro -- GPT-OSS 120B,0.8000,0.2625,0.0625,> +2σ,> +2σ
19,Llama Maverick -- GPT-OSS 120B,0.7125,0.2500,0.0875,+1σ to +2σ,> +2σ
17,Llama Maverick -- Claude Opus 4.5,0.6750,0.2125,0.0875,≥ mean,+1σ to +2σ
26,GPT Pro -- Qwen Max Thinking,0.6750,0.1875,0.0500,≥ mean,≥ mean
10,Gemini Pro -- Gemini Flash 3,0.6750,0.1125,0.0375,≥ mean,-1σ to -2σ
27,GPT Pro -- Llama Maverick,0.6500,0.2125,0.0500,≥ mean,+1σ to +2σ
3,Claude Opus 4.5 -- Claude Haiku 4.5,0.6500,0.1250,0.0875,≥ mean,< mean
4,Claude Opus 4.5 -- GPT-OSS 120B,0.6500,0.2000,0.0500,≥ mean,≥ mean
23,GPT Pro -- Claude Opus 4.5,0.6500,0.1750,0.0375,≥ mean,≥ mean
9,Qwen Max Thinking -- GPT-OSS 120B,0.6375,0.2000,0.0500,≥ mean,≥ mean
